# Parse QC/Quant Results

Reads search engine results from `QC_Quant`, parses them to ProteoBench format,
and saves `datapoints.pkl` and `intermediates.pkl`.

Already-parsed entries are loaded from a per-entry cache (`Data/ProteoBench/cache/`);
only new entries are processed. To force re-parsing, delete the relevant `.pkl` file(s)
from the cache directory.

**Directory structure under `QC_Quant`:**
```
QC_Quant/
  <lab>/           e.g. DTU_Astral_1
    <engine>/      e.g. DIANN, AlphaDIA, PEAKS ...
      <settings>/  integer key defined in constants.search_settings_names
```

**Variants generated per entry:**
- `main`      - standard result
- `ticnorm`   - TIC-normalised copy (PEAKS settings 1-3, Chimerys all, Spectronaut settings 5)
- `firstpass` - DIA-NN first-pass report (only DIANN settings 4)


## 1. Imports

In [4]:
import os
import pickle
from itertools import chain

import pandas as pd
from tqdm import tqdm

from SCI_py.constants import LAB_NAME_MAPPING

# Parsing helpers (per-engine readers, variant logic, cache) live in SCI_py.benchmark.
from SCI_py.benchmark import (
    parse_tables_to_proteobench,
    parse_fasta_headers,
    read_result_folder,
    read_diann_firstpass,
    apply_tic_norm,
    enumerate_entries,
    EntryCache,
)


## 2. Configuration

Paths used throughout the notebook:

- **`ROOT_QCQUANT`** - root of the search-engine output tree, organised as
  `<lab>/<engine>/<settings>` (see the directory structure above). Stored in PRIDE but should be structured accordingly.
- **`ROOT_RAW`** - root of the raw `.raw`/`.d` files. PEAKS does not store
  the filenames in its export, so the raw files (ordered) are needed
  to infer them (`read_data_peaks`). Is enough to have empty placeholder files there to perform the mapping for PEAKS.
- **`METADATA_PATH`** - sample-level metadata (`metadata.parquet`, from
  `0_Create_data_objects/01_metadata.ipynb`). Merged into every report to map
  each run to its sample annotation.
- **`FASTA_PATH`** - search database. Used to build an accession -> entry-name
  map that backfills `Protein.Names` in DIA-NN second-pass reports. Take from PRIDE.
- **`DATA_OUT`** - output directory for `datapoints.pkl` and
  `intermediates.pkl`.
- **`CACHE_DIR`** - per-entry parse cache, so new entries can be added without
  re-parsing the full dataset.

**`LABNAME_TO_RAW`** maps each lab to the directory containing its raw files
(only consulted by the PEAKS reader).

In [ ]:
ROOT_QCQUANT  = '/public/conode53/sam/SCI_paper/PRIDE_Results/QC_Quant'
ROOT_RAW      = '/public/conode53/sam/SCI'
METADATA_PATH = '/home/sam/SCI/SCI/Data/metadata/metadata.parquet'
FASTA_PATH    = '/public/conode53/sam/SCI/fasta/human_ecoli_yeast_contaminant_revi_2025_10_17.fasta'
DATA_OUT      = '/home/sam/SCI/SCI/Data/ProteoBench'
CACHE_DIR     = os.path.join(DATA_OUT, 'cache')

# EntryCache creates CACHE_DIR if it does not exist.
cache = EntryCache(CACHE_DIR)

# Maps lab name to the directory containing raw .raw/.d files (needed by PEAKS).
LABNAME_TO_RAW = {
    'DTU_Astral_1':    os.path.join(ROOT_RAW, 'Erwin',        'raw', 'mixture'),
    'LSMBO_Astral_1':  os.path.join(ROOT_RAW, 'Jeewan',       'raw', 'mixture'),
    'KI_Astral_1':     os.path.join(ROOT_RAW, 'KI_Akos',      'raw', 'mixture'),
    'Wyss_Astral_1':   os.path.join(ROOT_RAW, 'Bogdan',       'raw', 'mixture'),
    'IMP_Astral_1':    os.path.join(ROOT_RAW, 'Karl_plate_5', 'raw', 'mixture'),
    'KI_Astral_2':     os.path.join(ROOT_RAW, 'KI_Akos_11',   'raw', 'mixture'),
    'IMP_timsTOF_1':   os.path.join(ROOT_RAW, 'Karl_plate_6', 'raw', 'mixture'),
    'Broad_timsTOF_1': os.path.join(ROOT_RAW, 'Claudia',      'raw', 'mixture'),
    'BYU_timsTOF_1':   os.path.join(ROOT_RAW, 'Kelly',        'raw', 'mixture'),
}

## 3. Load metadata and FASTA mapping

In [7]:
metadata      = pd.read_parquet(METADATA_PATH)
fasta_mapping = parse_fasta_headers(FASTA_PATH)

print(f'Loaded {len(fasta_mapping):,} FASTA entries.')

Loaded 31,254 FASTA entries.


## 4. Parsing logic (imported)

The per-engine readers and variant logic now live in
`SCI_py.benchmark` and are imported above:

- **`read_result_folder(folder, lab, engine, settings, *, metadata, fasta_mapping, labname_to_raw)`**
  reads one result folder and returns a *table* dict with keys
  `report`, `user_input`, `comment`, `input_format`, `lab_name`. Each engine
  has its own reader, q-value filter, and quantity column; this function
  dispatches on `engine` and returns `None` for unhandled engines.
- **`read_diann_firstpass(folder, lab, settings, *, metadata)`** reads the
  DIA-NN `report-first-pass.parquet` instead of the second-pass report.
- **`apply_tic_norm(table)`** returns a copy of a table with TIC-normalised
  quantities (the normalised column depends on the engine).

The table dict is the unit consumed by `parse_tables_to_proteobench`.

## 5. Cache

Caching is handled by `EntryCache` (instantiated as `cache` in section 2).
Each parsed entry is stored as
`cache/{lab}__{engine}__{settings}__{variant}.pkl`, holding a dict with keys
`datapoint` (DataFrame) and `intermediates` (list of DataFrames).

Methods: `cache.is_cached(...)`, `cache.save(...)`, `cache.load(...)`.

To force an entry to be re-parsed, delete its `.pkl` file from the cache
directory.

## 6. Enumerate all entries

Walk the data directory and build the full list of
`(result_folder, lab, engine, settings, variant)` tuples to process.


In [8]:
all_entries = enumerate_entries(ROOT_QCQUANT)  # (result_folder, lab, engine, settings, variant)

n_cached = sum(
    cache.is_cached(lab, engine, settings, variant)
    for _, lab, engine, settings, variant in all_entries
)
print(
    f'{len(all_entries)} total entries  |  '
    f'{n_cached} already cached  |  '
    f'{len(all_entries) - n_cached} to parse'
)

75 total entries  |  0 already cached  |  75 to parse


## 7. Parse new entries

Cached entries are skipped. Each newly parsed entry is written to the cache immediately
so a partial run can be resumed without re-processing already-completed entries.


In [13]:
n_new = n_skip = n_fail = 0

for result_folder, lab, engine, settings, variant in tqdm(all_entries):
    if cache.is_cached(lab, engine, settings, variant):
        n_skip += 1
        continue

    # --- read raw data ---
    try:
        if variant == 'firstpass':
            table = read_diann_firstpass(result_folder, lab, settings, metadata=metadata)
        else:
            table = read_result_folder(
                result_folder, lab, engine, settings,
                metadata=metadata,
                fasta_mapping=fasta_mapping,
                labname_to_raw=LABNAME_TO_RAW,
            )
            if table is None:
                continue
            if variant == 'ticnorm':
                table = apply_tic_norm(table)
    except Exception as e:
        print(f'  [READ ERROR]  {lab}/{engine}/{settings} [{variant}]: {e}')
        n_fail += 1
        continue

    # --- parse to ProteoBench format ---
    try:
        dtp, intermed = parse_tables_to_proteobench(
            tables=[table],
            lab_name_mapping=LAB_NAME_MAPPING,
            default_cutoff_min_prec=3,
        )
    except Exception as e:
        print(f'  [PARSE ERROR] {lab}/{engine}/{settings} [{variant}]: {e}')
        n_fail += 1
        continue

    cache.save(lab, engine, settings, variant, dtp, intermed)
    n_new += 1

print(f'\nDone: {n_new} newly parsed  |  {n_skip} skipped (cached)  |  {n_fail} failed')

100%|██████████| 75/75 [10:27<00:00,  8.36s/it]


Done: 75 newly parsed  |  0 skipped (cached)  |  0 failed


## 8. Combine cached results

In [14]:
datapoints_list    = []
intermediates_list = []

for result_folder, lab, engine, settings, variant in all_entries:
    cached = cache.load(lab, engine, settings, variant)
    if cached is None:
        continue
    datapoints_list.append(cached['datapoint'])
    intermediates_list.append(cached['intermediates'])

datapoints    = pd.concat(datapoints_list,                  ignore_index=True)
intermediates = pd.concat(list(chain(*intermediates_list)), ignore_index=True)

print(f'{len(datapoints)} datapoints  |  {len(intermediates):,} intermediate rows')

75 datapoints  |  1,020,110 intermediate rows


## 9. Save

In [ ]:
with open(os.path.join(DATA_OUT, 'datapoints.pkl'), 'wb') as f:
    pickle.dump(datapoints, f)

with open(os.path.join(DATA_OUT, 'intermediates.pkl'), 'wb') as f:
    pickle.dump(intermediates, f)

print(f'Saved to {DATA_OUT}')
